# 何琪铭

# 1

In [ ]:
SELECT
    o.order_id,
    u.username,
    o.order_date,
    o.status,
    p.product_name,
    oi.quantity,
    oi.unit_price,
    oi.quantity * oi.unit_price AS total_price
FROM
    orders o
JOIN  users u ON o.user_id = u.user_id
JOIN  order_items oi ON o.order_id = oi.order_id
JOIN  products p ON oi.product_id = p.product_id
ORDER BY o.order_id ASC, oi.item_id ASC;

# 2

In [ ]:
SELECT
    p.category,
    SUM(oi.quantity) AS total_quantity,
    SUM(oi.quantity * oi.unit_price) AS total_sales,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM
    products p
JOIN  order_items oi ON p.product_id = oi.product_id
JOIN  orders o ON oi.order_id = o.order_id
WHERE o.status = '已完成'
GROUP BY p.category
HAVING SUM(oi.quantity * oi.unit_price) > 500
ORDER BY total_sales DESC;

# 3

In [ ]:
--查询订单总金额高于所有订单平均金额的订单，输出：订单ID、用户名、订单总金额、订单状态、下单时间
SELECT
    o.order_id,
    u.username,
    o.total_amount,
    o.status,
    o.order_date
FROM
    orders o
JOIN  users u ON o.user_id = u.user_id
WHERE o.total_amount > (
    SELECT AVG(total_amount) 
    FROM orders 
    )
ORDER BY o.total_amount DESC;

# 4

In [ ]:
# 更新前查询会被影响的商品
SELECT
    product_id,
    product_name,
    stock,
    status
FROM products
WHERE stock <= 45
AND status = 'A';

# 执行更新
UPDATE products
SET status = 'I'
WHERE stock <= 45
AND status = 'A';

# 检查更新结果
SELECT
    product_id,
    product_name,
    stock,
    status
FROM products
WHERE stock <= 45;

# 5

In [ ]:
SELECT
    u.user_id,
    u.username,
    u.vip_level,
    COUNT(o.order_id) AS efficient_order_count,
    SUM(o.total_amount) AS total_spent,
    MIN (o.order_date) AS first_order_date
FROM
    users u
    JOIN orders o ON u.user_id = o.user_id
WHERE o.status IN ('已完成', '已支付','已发货')
--AND o.order_date BETWEEN '2024-11-01' AND '2024-11-30'这样会有边界遗漏，用下面的方法更精细
AND o.order_date >= '2024-11-01 00:00:00' 
AND o.order_date < '2024-12-01 00:00:00'
GROUP BY u.user_id, u.username, u.vip_level
ORDER BY total_spent DESC
LIMIT 5;


# 陈慧欣

# 1

In [ ]:
--查询每个用户的总订单金额，显示用户ID、用户名以及总金额。要求必须包含所有用户，没下过单的用户也要显示，总金额为0。
SELECT
    u.user_id,
    u.username,
    COALESCE(SUM(o.total_amount), 0) AS total_spent
FROM
    users u
LEFT JOIN orders o ON u.user_id = o.user_id
GROUP BY u.user_id, u.username
ORDER BY total_spent DESC;

# 2

In [ ]:
--统计每个商品分类中 曾经被购买过的 商品数量，并只显示商品数量 > 2 的分类。要求必须使用 GROUP BY / HAVING，且不能使用子查询。
SELECT
    p.category,
    COUNT(DISTINCT p.product_id) AS purchased_product_count
FROM        
    products p
    JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.category
HAVING COUNT(DISTINCT p.product_id) > 2;

# 3

In [ ]:
--嵌套查询：找出那些购买过商品原始价格高于500元的客户ID。
SELECT DISTINCT user_id
FROM orders o
WHERE o.order_id IN (
    SELECT order_id
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    WHERE p.price > 500
);

# 4

In [ ]:
--将所有库存数量小于50的商品单价临时降价20%，即乘以0.8，以便清库存。
UPDATE products
SET price = price * 0.8
WHERE stock < 50
AND stock > 0; 

更新前后对比：
SELECT *
FROM products
WHERE stock < 50
AND stock > 0; 

# 5

In [ ]:
--查询2024年11月下单的所有订单，按订单总金额降序排列，取第6到第10条记录。显示订单号、用户姓名、订单日期、订单总金额。
SELECT
    o.order_id,
    u.username,
    o.order_date,
    o.total_amount
FROM
    orders o
JOIN  users u ON o.user_id = u.user_id
WHERE o.order_date >= '2024-11-01 00:00:00'
AND o.order_date < '2024-12-01 00:00:00'
ORDER BY o.total_amount DESC
LIMIT 5 OFFSET 5; -- OFFSET 5 跳过前5条记录，从第6条开始显示5条记录
--考虑到可能有多条订单金额相同的情况，使用下面的方法更精确地获取第6到第10条记录
WITH RankedOrders AS (
    SELECT
        o.order_id,
        u.username,
        o.order_date,
        o.total_amount,
        ROW_NUMBER() OVER (ORDER BY o.total_amount DESC) AS rn
    FROM
        orders o
    JOIN  users u ON o.user_id = u.user_id
    WHERE o.order_date >= '2024-11-01 00:00:00'
    AND o.order_date < '2024-12-01 00:00:00'
)
SELECT order_id, username, order_date, total_amount
FROM RankedOrders
WHERE rn BETWEEN 6 AND 10
ORDER BY total_amount DESC;